# EXPRES solar: an RV time series

Here we query the EXPRES solar Level-2 products over a date range and plot the
radial-velocity series with its error bars — a dense, high-cadence look at the
Sun observed as a star.

```{note}
This notebook runs against the live NEID/EXPRES server. The published version
here uses synthetic data so it builds without an archive account — the numbers
shown are illustrative, not real observations.
```

## What is EXPRES solar data?

{term}`EXPRES`, the EXtreme PREcision Spectrograph, sits on the Lowell Discovery
Telescope. Like NEID it has a small {term}`solar feed` — a telescope that feeds
sunlight into the spectrograph during the day, observing the Sun as if it were a
distant star. This client exposes **only** that solar data, and only at the
solar L1 and L2 levels (it is hosted on the same IPAC archive as NEID). The
result is a long, closely-spaced {term}`radial velocity (RV)` time series, used
to study solar activity and to validate the instrument's stability. See
{doc}`../concepts/instruments` for more on both spectrographs.

EXPRES solar data is fully **public** — no login required.

In [ ]:
import os

import numpy as np
from astropy.table import Table

OFFLINE = bool(os.getenv("NEXSCI_RV_OFFLINE_FIXTURES"))

if OFFLINE:
    from unittest.mock import patch

    rng = np.random.default_rng(42)
    n = 30
    base_mjd = 60615.5
    _fake = Table(
        {
            "l2filename": [f"expressolar_2024110{i // 10}_{i % 10:04d}.fits" for i in range(n)],
            "obsdate": [
                f"2024-11-{1 + i // 10:02d}T{14 + (i % 10) // 4:02d}:{15 + (i % 10) * 5:02d}:00"
                for i in range(n)
            ],
            "mjd": base_mjd + np.arange(n) * 0.01,
            "v": rng.normal(0.0, 1.2, n),
            "e_v": rng.uniform(0.5, 0.9, n),
            "dvrms": rng.uniform(0.4, 0.7, n),
            "extsnr": rng.uniform(180, 320, n),
        }
    )
    _patcher = patch("nexsci_rv.tap.TapClient.query", return_value=_fake)
    _patcher.start()
    print("Offline mode: queries return synthetic data.")

## Build the archive client

In [ ]:
from nexsci_rv import EXPRESArchive

expres = EXPRESArchive()
expres.data_levels

`expres.data_levels` lists the two levels EXPRES offers:

- `solarl1` — the calibrated 1D {term}`extracted spectrum` (flux vs.
  wavelength), for your own line or activity analysis.
- `solarl2` — the RV-ready products, where the radial velocity already lives.

We use `solarl2` below because we want the velocities, not the raw spectra. See
{doc}`../concepts/data-levels` for the full level ladder.

## Query the solar L2 RV table

In [ ]:
sun = expres.query_datetime(
    "solarl2",
    ("2024-11-01", "2024-11-06"),
    columns=["l2filename", "obsdate", "mjd", "v", "e_v", "dvrms", "extsnr"],
    maxrec=200,
)
sun[:5]

The columns we kept, for EXPRES solar L2:

- `l2filename` — the L2 {term}`FITS` file name (the {term}`download` key).
- {term}`date_obs / obsdate` — the observation time stamp; on EXPRES the
  datetime column is `obsdate` (not `date_obs`), and {term}`query_datetime`
  routes through it automatically.
- `mjd` — the observation time as a {term}`Julian Date (JD / MJD)`; the natural
  numeric x-axis for a time series.
- {term}`v / e_v (EXPRES)` — **the answer**: `v` is the radial velocity in m/s
  and `e_v` its uncertainty. These play the role {term}`ccfrvmod` and
  {term}`ccfnoise` play on NEID.
- {term}`dvrms (EXPRES)` — the photon-noise precision floor for the exposure.
- {term}`extsnr (EXPRES)` — extracted signal-to-noise; a natural quality cut
  (e.g. `where='extsnr > 200'`).

Full column tour: {doc}`../concepts/results-columns`. As with NEID, `maxrec`
caps the returned rows (200 here); raise it once a query looks right.

```{tip}
`mjd` is a plain number. To convert it to a calendar date or another time
scale, wrap it in `astropy.time.Time(sun['mjd'], format='mjd')`.
```

## Plot the RV time series

`v` is the modelled radial velocity, `e_v` its formal uncertainty, and
`dvrms` the photon-noise floor.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(
    sun["mjd"],
    sun["v"],
    yerr=sun["e_v"],
    fmt="o",
    ms=3,
    capsize=2,
    alpha=0.7,
)
ax.set_xlabel("MJD")
ax.set_ylabel("RV (m/s)")
ax.set_title("EXPRES solar RV — 2024-11-01 to 2024-11-06")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## Why some searches don't apply

On a normal target you can search by sky position, by name, or by catalog
ID. The Sun has no fixed celestial coordinate, so EXPRES solar rows carry
{term}`sentinel value` coordinates (`-99.9`) instead of a real RA/Dec. The
package guards against treating those as a real position: position-, object-,
and qobject-based searches raise {py:exc}`~nexsci_rv.exceptions.NotApplicable`.
Reach for {term}`query_datetime` (optionally narrowed with a `where=` cut)
instead.

Here is what that guard looks like in practice:

In [ ]:
from astropy import units as u
from astropy.coordinates import SkyCoord

from nexsci_rv import NotApplicable

try:
    expres.query_position("solarl2", SkyCoord("0d", "0d"), 1 * u.deg)
except NotApplicable as exc:
    print(f"Got expected NotApplicable: {exc}")

## What next

- {doc}`03_download_and_plot` — download FITS files and open a spectrum.
- {doc}`../archives/expres` — the full catalog of EXPRES solar quirks.
- {doc}`../concepts/choosing-a-query` — picking a query method per archive.